In [20]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

In [21]:
x, y, z = sp.symbols('x y z')
f = -sp.cos(x) - sp.cos(y) - sp.cos(z)

In [22]:
H = sp.hessian(f, (x, y, z))
print(H)

Matrix([[cos(x), 0, 0], [0, cos(y), 0], [0, 0, cos(z)]])


In [23]:
# We cannot use Sympy's diagonalize() because it does not find an orthogonal Q
# Instead, we perform manual orthogonal diagonalization

# Find the eigenvalues of the 2x2 matrix
eigs = H.eigenvals(multiple=True)
lamda_1 = eigs[0]
lamda_2 = eigs[1]
lamda_3 = eigs[2]
lamda = sp.diag(*eigs)
print(lamda)

Matrix([[cos(x), 0, 0], [0, cos(y), 0], [0, 0, cos(z)]])


In [24]:
# Find the positive and negative components of the eigenvalues
a = sp.symbols('a')
g = lambda a: sp.simplify((a + sp.Abs(a)) / 2)
h = lambda a: sp.simplify(((- a) + sp.Abs(- a)) / 2)

lamda_plus = lamda.applyfunc(g)
lamda_minus = lamda.applyfunc(h)

print(lamda_minus)
print(lamda_plus)

Matrix([[-cos(x)/2 + Abs(cos(x))/2, 0, 0], [0, -cos(y)/2 + Abs(cos(y))/2, 0], [0, 0, -cos(z)/2 + Abs(cos(z))/2]])
Matrix([[cos(x)/2 + Abs(cos(x))/2, 0, 0], [0, cos(y)/2 + Abs(cos(y))/2, 0], [0, 0, cos(z)/2 + Abs(cos(z))/2]])


In [37]:
# Use the characteristic polynomials (H-lambda_1*I)x_1 = 0 (H-lambda_2*I)x_2 = 0
# Find the eigenvectors

b = sp.symbols('b')
j = lambda b: (H - b * sp.eye(H.shape[0])).nullspace()

X = [j(lamda_1)[0], j(lamda_2)[0], j(lamda_3)[0]]

print(X)


[Matrix([
[1],
[0],
[0]]), Matrix([
[0],
[1],
[0]]), Matrix([
[0],
[0],
[1]])]


In [39]:
# Use the Gram-Schmidt process to orthonormalize the vectors
# Write my own custom process for 3x3 to keep things simple

# u is a 2x1 vector
def normalize(u):
    numerator = u
    denominator = sp.sqrt(u[0] ** 2 + u[1] ** 2 + u[2] ** 2)
    return numerator / denominator

# u, v are 2x1 vectors
def proj(u, v):
    numerator = v.dot(u)
    denominator = u.dot(u)
    return (numerator / denominator) * u

u_1 = X[0]
e_1 = normalize(u_1)

u_2 = (X[1] - proj(u_1, X[1]))
e_2 = normalize(u_2)

u_3 = X[2] - proj(u_1, X[2]) - proj(u_2, X[2])
e_3 = normalize(u_3)

print(e_1)
print(e_2)
print(e_3)
print(e_1.dot(e_2), e_2.dot(e_3), e_3.dot(e_1))

Matrix([[1], [0], [0]])
Matrix([[0], [1], [0]])
Matrix([[0], [0], [1]])
0 0 0


In [40]:
# Put these orthonormal vectors into the columns of a matrix
# Need to do some hoop-jumping as Sympy matrices are defined by row

Q = sp.Matrix([e_1.T, e_2.T, e_3.T]).T
print(Q.shape)

(3, 3)


In [41]:
print(H)
print(Q * lamda * Q.T)

Matrix([[cos(x), 0, 0], [0, cos(y), 0], [0, 0, cos(z)]])
Matrix([[cos(x), 0, 0], [0, cos(y), 0], [0, 0, cos(z)]])


In [42]:
H_minus = Q * lamda_minus * Q.T 
H_plus = Q * lamda_plus * Q.T

print(H)
print(H_plus - H_minus)

Matrix([[cos(x), 0, 0], [0, cos(y), 0], [0, 0, cos(z)]])
Matrix([[cos(x), 0, 0], [0, cos(y), 0], [0, 0, cos(z)]])


In [43]:
# X is a 2x2 Sympy matrix
def nuclear_norm(X):
    eigs = X.eigenvals(multiple=True)
    abs_eigs = map(abs, eigs)
    return sum(abs_eigs)

print(nuclear_norm(H))
print(nuclear_norm(H_minus))
print(nuclear_norm(H_plus))
print(nuclear_norm(H_plus) + nuclear_norm(H_minus))

Abs(cos(x)) + Abs(cos(y)) + Abs(cos(z))
Abs(cos(x)/2 - Abs(cos(x))/2) + Abs(cos(y)/2 - Abs(cos(y))/2) + Abs(cos(z)/2 - Abs(cos(z))/2)
Abs(cos(x)/2 + Abs(cos(x))/2) + Abs(cos(y)/2 + Abs(cos(y))/2) + Abs(cos(z)/2 + Abs(cos(z))/2)
Abs(cos(x)/2 - Abs(cos(x))/2) + Abs(cos(x)/2 + Abs(cos(x))/2) + Abs(cos(y)/2 - Abs(cos(y))/2) + Abs(cos(y)/2 + Abs(cos(y))/2) + Abs(cos(z)/2 - Abs(cos(z))/2) + Abs(cos(z)/2 + Abs(cos(z))/2)
